SILVER

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

df_clientes = spark.read.table("bronze.default.clientes_bronze")\
  .dropDuplicates(['cliente_id'])
df_estoque = spark.read.table("bronze.default.estoque_bronze")\
  .dropDuplicates(['produto_id'])
df_vendas = spark.read.table("bronze.default.vendas_bronze")\
  .dropDuplicates(['venda_id'])

# Função para padronização de datas 
def padronizar_data(df, coluna):
    return df.withColumn(
        coluna,
        coalesce(
            try_to_date(col(coluna), "yyyy-MM-dd"),   
            try_to_date(col(coluna), "yyyy/MM/dd"),   
            try_to_date(col(coluna), "dd/MM/yyyy"),   
            try_to_date(col(coluna), "d/M/yyyy"),     
            try_to_date(col(coluna), "d/M/yy")        
        )
    )

# Padronizar datas
df_clientes = padronizar_data(df_clientes, "data_cadastro")
df_estoque = padronizar_data(df_estoque, "data_cadastro")
df_vendas = padronizar_data(df_vendas, "data_venda")
df_vendas = padronizar_data(df_vendas, "data_despesa")

# Função criar coluna ano_mes
def cria_coluna_ano_mes(df, coluna):
    return df.withColumn(
        "ano_mes"+ "_" + (coluna),
        concat(
            year(col(coluna)).cast("string"),
            lit("-"),
            month(col(coluna)).cast("string")
        )
    )

# Criação ano_mes
df_vendas = cria_coluna_ano_mes(df_vendas, "data_venda")
df_vendas = cria_coluna_ano_mes(df_vendas, "data_despesa")  
df_clientes = cria_coluna_ano_mes(df_clientes, "data_cadastro")
df_estoque = cria_coluna_ano_mes(df_estoque, "data_cadastro")

# Função para alterar valores monetários
def padronizar_valor(df, coluna):
    return df.withColumn(
        coluna,
        regexp_replace(
            regexp_replace(
                trim(col(coluna)),
                "R\\$\\s*", ""),   
            ",", ".")             
        .cast("decimal(10,2)")
    )

# Alterar valores monetários
df_estoque = padronizar_valor(df_estoque, "custo_producao")
df_estoque = padronizar_valor(df_estoque, "preco_venda")
df_vendas = padronizar_valor(df_vendas, "preco_unitario")
df_vendas = padronizar_valor(df_vendas, "valor_despesa")    

# Função tratar valores inteiros nulos
def tratar_valores_inteiros_nulos(df, coluna):
    return df.withColumn(
        coluna,
        coalesce(
            regexp_replace(col(coluna), "[^0-9]", "").cast("int"),
            lit(0)
        )
    )
# Tratar valores inteiros nulos
df_estoque = tratar_valores_inteiros_nulos(df_estoque, "quantidade_disponivel")
df_vendas = tratar_valores_inteiros_nulos(df_vendas, "quantidade")
df_vendas = tratar_valores_inteiros_nulos(df_vendas, "desconto_percent")         

# Criar coluna valor_liquido = Preco_unitario - desconto_percent
df_vendas = df_vendas.withColumn(
    "valor_liquido",
    col("preco_unitario") * (1 - col("desconto_percent")/100)
)

# Criar coluna valor_total = valor_liquido * quantidade
df_vendas = df_vendas.withColumn(
    "valor_total",
    col("valor_liquido") * col("quantidade")
)

# Criar coluna valor_total_despesa = valor_despesa * quantidade
df_vendas = df_vendas.withColumn(
    "valor_total_despesa",
    col("valor_despesa") * col("quantidade")
)

email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]{2,}\.[A-Za-z]{2,}$'

df_clientes = df_clientes.withColumn(
    "email_valido",
    when(col("email").isNull(), False)
    .otherwise(col("email").rlike(email_regex))
)

df_clientes = df_clientes.withColumn(
    "canal_aquisicao",
    when(
        col("canal_aquisicao").isNull() |
        (trim(col("canal_aquisicao")) == "") |
        col("canal_aquisicao").rlike("^[0-9]{4}-[0-9]{2}-[0-9]{2}$"),  # remove datas
        "Nao Informado"
    ).otherwise(initcap(trim(col("canal_aquisicao"))))
)

df_vendas = df_vendas.join(
    df_clientes.select("cliente_id", "canal_aquisicao"),
    on="cliente_id",
    how="left"
)

df_vendas = df_vendas.withColumn(
    "registro_valido",
    col("venda_id").isNotNull() &
    col("cliente_id").isNotNull() &
    col("produto_id").isNotNull() &
    col("data_venda").isNotNull() &
    col("preco_unitario").isNotNull() &
    (col("quantidade").cast("int") > 0)
)


df_estoque = df_estoque.withColumn(
    "registro_valido",
    col("produto_id").isNotNull() &
    col("nome_produto").isNotNull() &
    col("categoria").isNotNull() &
    col("custo_producao").isNotNull() &
    col("preco_venda").isNotNull() &
    (col("quantidade_disponivel").cast("int") >= 0)
)

df_clientes = df_clientes.withColumn(
    "registro_valido",
    col("cliente_id").isNotNull() &
    col("nome_cliente").isNotNull() &
    col("telefone").isNotNull() &
    col("cidade").isNotNull() &
    col("estado").isNotNull() &
    col("data_cadastro").isNotNull() &
    col("canal_aquisicao").isNotNull() &
    col("email_valido") == True
)

# Tabelas Inválidas. 
df_clientes_invalidos = df_clientes.filter(
    col("canal_aquisicao") == "Nao Informado"
).select(
    lit("clientes").alias("origem_tabela"),
    col("cliente_id").alias("id_registro"),
    lit("canal_aquisicao").alias("coluna_erro"),
    col("canal_aquisicao").alias("valor_original"),
    lit("Valor inválido ou ausente").alias("motivo_erro"),
    current_timestamp().alias("data_processo")
)

df_vendas_invalidos = df_vendas.filter(
    col("registro_valido") == False
).select(
    lit("vendas").alias("origem_tabela"),
    col("venda_id").alias("id_registro"),
    lit("registro_completo").alias("coluna_erro"),
    lit(None).alias("valor_original"),
    lit("Campos obrigatórios inválidos").alias("motivo_erro"),
    current_timestamp().alias("data_processo")
)

df_estoque_invalidos = df_estoque.filter(
    col("registro_valido") == False
).select(
    lit("estoque").alias("origem_tabela"),
    col("produto_id").alias("id_registro"),
    lit("registro_completo").alias("coluna_erro"),
    lit(None).alias("valor_original"),
    lit("Campos obrigatórios inválidos").alias("motivo_erro"),
    current_timestamp().alias("data_processo")
)

df_invalidos = df_clientes_invalidos \
    .unionByName(df_vendas_invalidos) \
    .unionByName(df_estoque_invalidos)


# Salvar tabelas
df_clientes.write \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.default.clientes_silver")

df_estoque.write \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.default.estoque_silver")

df_vendas.write \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.default.vendas_silver")

df_invalidos.write \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.default.registros_invalidos")

display(df_clientes.limit(10))
display(df_estoque.limit(10))
display(df_vendas.limit(10))
display(df_invalidos.limit(10))